# 03 — Temporal Holdout: Persistence Filters and Chronological Out-of-Sample Test

**Project:** Volatility & Dispersion (V&D) Cloud — personal research project
**Phase:** 2B — tickets G10–G11
**Data:** public daily prices via `yfinance` only (same developed-market country ETF surface as
`02_cross_market_validation_clean.ipynb`)

**Why this notebook.** `02_cross_market_validation_clean.ipynb` ended with a mixed, unresolved
verdict: descriptive regime separation existed, but it didn't clearly survive a baseline challenge
or a stacked permutation-plus-HAC robustness check. Two questions were left open:

1. Does requiring a regime to *persist* for several consecutive days before counting it — rather
   than using same-date, potentially flickery raw labels — produce a cleaner, more stable picture?
2. Do any of the relationships surfaced in-sample (2015–2020) actually hold up on data the
   framework never saw during construction (2021–2026)?

This notebook answers both, in that order, and closes with the honest final verdict of the
research arc.

**Clean-room note.** Public market data only. No proprietary employer systems, employer data,
internal tools, or internal methodology are referenced or reproduced anywhere in this notebook.

**Curation note.** This is a compressed retelling of the private working notebook's G10–G11
tickets. The logic here is the actual logic used, cleaned up for readability. It has **not been
re-executed** in this curation pass — run `pip install -r requirements.txt` and execute
top-to-bottom fresh (starting from notebook 02's data-ingestion and cloud-geometry steps) before
treating any printed number as current.

## 1. Rebuild the G1–G4 Foundation

This notebook assumes the same EFA / URTH / 13-country-ETF surface, signal variants, and same-date
cloud geometry built in `02_cross_market_validation_clean.ipynb` (tickets G1–G4). The setup is
repeated here in compressed form so this notebook can run standalone.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf

warnings.filterwarnings("ignore")

sys.path.insert(0, str(Path("..").resolve()))
import config

PRIMARY_BENCHMARK = config.DM_PRIMARY_BENCHMARK   # EFA
COMPONENT_TICKERS = config.DM_COUNTRY_ETF_UNIVERSE
ALL_TICKERS = [PRIMARY_BENCHMARK, config.DM_CONTEXT_BENCHMARK] + COMPONENT_TICKERS

Z_WINDOW, VOL_WINDOW, SMOOTH_WINDOW = config.Z_WINDOW, config.VOL_WINDOW, config.SMOOTH_WINDOW
CLOUD_FAST, CLOUD_MEDIUM, CLOUD_SLOW = config.CONVERSION_WINDOW, config.BASE_WINDOW, config.SPAN_B_WINDOW
FORWARD_HORIZONS = config.FORWARD_HORIZONS
MIN_BUCKET_N = config.MIN_BUCKET_N
PERSISTENCE_WINDOWS = config.PERSISTENCE_WINDOWS   # [3, 5]

raw = yf.download(tickers=ALL_TICKERS, start=config.START_DATE, end=config.END_DATE,
                   auto_adjust=True, progress=True)
close_field = "Close" if "Close" in raw.columns.get_level_values(0) else "Adj Close"
prices = raw[close_field].copy().sort_index().dropna(how="all")
returns = prices.pct_change().dropna(how="all")

efa_returns = returns[PRIMARY_BENCHMARK].copy()
dm_country_returns = returns[COMPONENT_TICKERS].copy()
available_count = dm_country_returns.notna().sum(axis=1)

ANNUALIZATION_FACTOR = np.sqrt(252)
dm_country_dispersion_ann = (dm_country_returns.std(axis=1, skipna=True, ddof=1)
                              .where(available_count >= 10)) * ANNUALIZATION_FACTOR
efa_realized_vol = efa_returns.rolling(window=VOL_WINDOW, min_periods=VOL_WINDOW).std(ddof=1) * ANNUALIZATION_FACTOR


def rolling_zscore(series, window):
    m = series.rolling(window=window, min_periods=window).mean()
    s = series.rolling(window=window, min_periods=window).std(ddof=1)
    return ((series - m) / s).replace([np.inf, -np.inf], np.nan)


dm_country_dispersion_z_smooth = rolling_zscore(dm_country_dispersion_ann, Z_WINDOW).rolling(window=SMOOTH_WINDOW, min_periods=SMOOTH_WINDOW).mean()
efa_realized_vol_z_smooth = rolling_zscore(efa_realized_vol, Z_WINDOW).rolling(window=SMOOTH_WINDOW, min_periods=SMOOTH_WINDOW).mean()

signal_variants = {
    "additive": 0.70 * dm_country_dispersion_z_smooth + 0.30 * efa_realized_vol_z_smooth,
    "penalty": 0.70 * dm_country_dispersion_z_smooth - 0.30 * efa_realized_vol_z_smooth,
    "dispersion_only": dm_country_dispersion_z_smooth.copy(),
    "volatility_only": efa_realized_vol_z_smooth.copy(),
}


def rolling_midpoint(series, window):
    high = series.rolling(window=window, min_periods=window).max()
    low = series.rolling(window=window, min_periods=window).min()
    return ((high + low) / 2.0).replace([np.inf, -np.inf], np.nan)


def build_same_date_cloud_regime(signal):
    conversion, base = rolling_midpoint(signal, CLOUD_FAST), rolling_midpoint(signal, CLOUD_MEDIUM)
    span_a = (conversion + base) / 2.0
    span_b = rolling_midpoint(signal, CLOUD_SLOW)
    cloud_top = pd.concat([span_a, span_b], axis=1).max(axis=1, skipna=False)
    cloud_bottom = pd.concat([span_a, span_b], axis=1).min(axis=1, skipna=False)
    valid = signal.notna() & cloud_top.notna() & cloud_bottom.notna()
    regime = pd.Series(np.where(signal > cloud_top, "above_cloud",
                        np.where(signal < cloud_bottom, "below_cloud", "inside_cloud")),
                        index=signal.index, dtype=object).where(valid)
    return regime.dropna()


raw_regimes = {name: build_same_date_cloud_regime(s) for name, s in signal_variants.items()}


def compute_forward_return(return_series, horizon):
    gross = (1.0 + return_series.shift(-1))
    cumulative = gross.rolling(window=horizon, min_periods=horizon).apply(np.prod, raw=True)
    return (cumulative - 1.0).shift(-(horizon - 1)).replace([np.inf, -np.inf], np.nan)


efa_fwd = pd.DataFrame({f"fwd_{h}d": compute_forward_return(efa_returns, h) for h in FORWARD_HORIZONS})

print("Foundation rebuilt.")
for name, s in raw_regimes.items():
    print(f"  {name:18s} valid regime obs: {s.notna().sum()}")

[                       0%                       ]

[******                13%                       ]  2 of 15 completed

[*************         27%                       ]  4 of 15 completed

[****************      33%                       ]  5 of 15 completed

[**********************47%                       ]  7 of 15 completed

...

[**********************80%*************          ]  12 of 15 completed

[**********************87%*****************      ]  13 of 15 completed

[**********************93%********************   ]  14 of 15 completed

[*********************100%***********************]  15 of 15 completed

Foundation rebuilt.
  additive           valid regime obs: 2667
  penalty            valid regime obs: 2667
  dispersion_only    valid regime obs: 2687
  volatility_only    valid regime obs: 2667


## 2. Persistence-Confirmation Logic (G10)

Same-date regime labels can flicker day to day. `create_persistence_confirmed_regime` labels date
*t* with the raw regime only if that same regime has held for the prior `window - 1` trading days
**and** date *t* itself — i.e. `window` consecutive identical, valid raw-regime observations ending
at *t*. Any gap, change, or missing value inside that window yields `"unconfirmed"`. Confirmed
labels are **not** forward-filled into unconfirmed dates — this is a strict, same-or-no-label
confirmation rule.

Two windows are tested: 3-day and 5-day confirmation.

In [2]:
UNCONFIRMED_LABEL = "unconfirmed"
REGIME_STATES = ["above_cloud", "inside_cloud", "below_cloud"]


def create_persistence_confirmed_regime(regime_series, window):
    values = regime_series.tolist()
    out = []
    for i in range(len(values)):
        if i < window - 1:
            out.append(UNCONFIRMED_LABEL)
            continue
        window_vals = values[i - window + 1:i + 1]
        first = window_vals[0]
        if first in REGIME_STATES and all(v == first for v in window_vals):
            out.append(first)
        else:
            out.append(UNCONFIRMED_LABEL)
    return pd.Series(out, index=regime_series.index)


confirmed_regimes = {}
for name, raw_series in raw_regimes.items():
    for window in PERSISTENCE_WINDOWS:
        confirmed_regimes[(name, window)] = create_persistence_confirmed_regime(raw_series, window)

# Worked example matching the confirmation rule
example_a = pd.Series(["above_cloud", "above_cloud", "above_cloud"])
example_b = pd.Series(["above_cloud", "above_cloud", "inside_cloud"])
print("3D example (above, above, above)  ->", create_persistence_confirmed_regime(example_a, 3).tolist())
print("3D example (above, above, inside) ->", create_persistence_confirmed_regime(example_b, 3).tolist())

3D example (above, above, above)  -> ['unconfirmed', 'unconfirmed', 'above_cloud']
3D example (above, above, inside) -> ['unconfirmed', 'unconfirmed', 'unconfirmed']


In [3]:
churn_rows = []
for name, raw_series in raw_regimes.items():
    raw_transitions = raw_series.ne(raw_series.shift()).sum()
    raw_rate = 100.0 * raw_transitions / len(raw_series)
    churn_rows.append({"variant": name, "regime_source": "raw", "transition_rate_per_100_obs": raw_rate,
                        "percent_unconfirmed": 0.0})
    for window in PERSISTENCE_WINDOWS:
        confirmed = confirmed_regimes[(name, window)]
        transitions = confirmed.ne(confirmed.shift()).sum()
        rate = 100.0 * transitions / len(confirmed)
        pct_unconfirmed = 100.0 * (confirmed == UNCONFIRMED_LABEL).mean()
        churn_rows.append({"variant": name, "regime_source": f"confirmed_{window}d",
                            "transition_rate_per_100_obs": rate, "percent_unconfirmed": pct_unconfirmed})

churn_diagnostics = pd.DataFrame(churn_rows)
churn_diagnostics

,variant,regime_source,transition_rate_per_100_obs,percent_unconfirmed
0,additive,raw,19.910011,0.000000
1,additive,confirmed_3d,19.797525,32.808399
2,additive,confirmed_5d,14.098238,50.956130
3,penalty,raw,22.234721,0.000000
4,penalty,confirmed_3d,22.309711,36.820397
5,penalty,confirmed_5d,15.785527,57.030371
6,dispersion_only,raw,24.674358,0.000000
7,dispersion_only,confirmed_3d,23.446223,39.523632
8,dispersion_only,confirmed_5d,15.742464,60.364719
9,volatility_only,raw,8.136483,0.000000


Persistence confirmation reduces churn (fewer regime transitions per 100 observations), as
expected by construction — but at a direct cost in usable sample size, since every date inside an
unbroken run of the required length gets folded into `"unconfirmed"` until it qualifies. Whether
that trade-off actually buys anything in terms of a cleaner or more temporally stable
forward-return relationship is the open empirical question G10/G11 exist to answer — it is not
assumed here.

In [4]:
def summarize_regime_forward_returns(regime_series, return_frame, source_label, variant_label):
    joined = pd.DataFrame({"regime_state": regime_series}).join(return_frame, how="left")
    rows = []
    for h in FORWARD_HORIZONS:
        col = f"fwd_{h}d"
        uncond_mean = joined[col].dropna().mean()
        for regime, group in joined.dropna(subset=["regime_state", col]).groupby("regime_state"):
            r = group[col]
            rows.append({"variant": variant_label, "regime_source": source_label, "regime": regime,
                         "horizon": h, "count": len(r), "mean_return": r.mean(),
                         "excess_vs_unconditional": r.mean() - uncond_mean})
    return rows


persistence_forward_rows = []
for name, raw_series in raw_regimes.items():
    persistence_forward_rows += summarize_regime_forward_returns(raw_series, efa_fwd, "raw", name)
    for window in PERSISTENCE_WINDOWS:
        confirmed = confirmed_regimes[(name, window)]
        persistence_forward_rows += summarize_regime_forward_returns(confirmed, efa_fwd, f"confirmed_{window}d", name)

persistence_forward_return_summary = pd.DataFrame(persistence_forward_rows)
persistence_forward_return_summary.head(12)

,variant,regime_source,regime,horizon,count,mean_return,excess_vs_unconditional
0,additive,raw,above_cloud,5,884,0.003064,0.001197
1,additive,raw,below_cloud,5,1244,0.001310,-0.000557
2,additive,raw,inside_cloud,5,534,0.001184,-0.000683
3,additive,raw,above_cloud,10,884,0.005875,0.002178
4,additive,raw,below_cloud,10,1239,0.002661,-0.001037
5,additive,raw,inside_cloud,10,534,0.002498,-0.001200
6,additive,raw,above_cloud,21,884,0.011995,0.004215
7,additive,raw,below_cloud,21,1232,0.007049,-0.000732
8,additive,raw,inside_cloud,21,530,0.002453,-0.005328
9,additive,raw,above_cloud,63,876,0.029790,0.006497


**G10 recommendation for G11.** Comparing raw, 3-day-confirmed, and 5-day-confirmed regimes
lens-by-lens (weighing churn reduction against the sample-size cost, not simply picking whichever
set happened to show the largest in-sample excess — that would just be overfitting the label
choice) did not produce a clean, uniform winner. Some lenses looked marginally cleaner under 5-day
confirmation; others lost too much sample size to be usable. **No variant showed a decisive,
broad-based improvement from persistence confirmation at the descriptive stage.** Both `raw` and
`confirmed_5d` regimes are carried forward into the temporal holdout test below, so the holdout
itself — not an in-sample preference — gets to arbitrate.

## 3. Chronological Holdout Split (G11)

Every date is labeled by period:

- **In-sample**: 2015-01-02 through 2020-12-31 — the period the framework was built and tuned
  against.
- **Holdout**: 2021-01-01 through the latest available date — data the framework never saw during
  construction.

For every `regime_source` (`raw`, `confirmed_5d`) × `variant` × `regime` × horizon combination, the
in-sample excess-vs-unconditional and the holdout excess-vs-unconditional are computed
**separately, each against that period's own unconditional mean** — so a "consistent" finding
can't just be an artifact of the whole market drifting in one direction between periods.

In [5]:
IN_SAMPLE_END = pd.Timestamp(config.IN_SAMPLE_END)
HOLDOUT_START = pd.Timestamp(config.HOLDOUT_START)

period = pd.Series(np.where(efa_fwd.index <= IN_SAMPLE_END, "in_sample",
                    np.where(efa_fwd.index >= HOLDOUT_START, "holdout", "excluded")),
                    index=efa_fwd.index)

print(period.value_counts())
print(f"\nIn-sample : {efa_fwd.index[period == 'in_sample'].min().date()} to {efa_fwd.index[period == 'in_sample'].max().date()}")
print(f"Holdout   : {efa_fwd.index[period == 'holdout'].min().date()} to {efa_fwd.index[period == 'holdout'].max().date()}")

in_sample    1510
holdout      1357
Name: count, dtype: int64

In-sample : 2015-01-05 to 2020-12-31
Holdout   : 2021-01-04 to 2026-05-29


In [6]:
def period_regime_forward_summary(regime_series, source_label, variant_label):
    joined = pd.DataFrame({"regime_state": regime_series, "period": period}).join(efa_fwd, how="left")
    rows = []
    for p in ["in_sample", "holdout"]:
        sub = joined[joined["period"] == p]
        for h in FORWARD_HORIZONS:
            col = f"fwd_{h}d"
            uncond_mean = sub[col].dropna().mean()
            for regime, group in sub.dropna(subset=["regime_state", col]).groupby("regime_state"):
                r = group[col]
                rows.append({"period": p, "regime_source": source_label, "variant": variant_label,
                             "regime": regime, "horizon": h, "count": len(r),
                             "excess_vs_unconditional": r.mean() - uncond_mean})
    return rows


period_forward_rows = []
for name, raw_series in raw_regimes.items():
    period_forward_rows += period_regime_forward_summary(raw_series, "raw", name)
    period_forward_rows += period_regime_forward_summary(confirmed_regimes[(name, 5)], "confirmed_5d", name)

regime_period_forward_return_summary = pd.DataFrame(period_forward_rows)
print(f"Total in-sample/holdout cells: {len(regime_period_forward_return_summary)}")

Total in-sample/holdout cells: 224


In [7]:
NEAR_ZERO_EXCESS = 1e-4

def lookup_period_cell(source, variant, regime, horizon, p):
    row = regime_period_forward_return_summary[
        (regime_period_forward_return_summary["period"] == p)
        & (regime_period_forward_return_summary["regime_source"] == source)
        & (regime_period_forward_return_summary["variant"] == variant)
        & (regime_period_forward_return_summary["regime"] == regime)
        & (regime_period_forward_return_summary["horizon"] == horizon)
    ]
    if row.empty:
        return None, None
    r = row.iloc[0]
    return int(r["count"]), float(r["excess_vs_unconditional"])


consistency_rows = []
for source in ["raw", "confirmed_5d"]:
    for variant in signal_variants:
        candidate_regimes = REGIME_STATES + ([UNCONFIRMED_LABEL] if source != "raw" else [])
        for regime in candidate_regimes:
            for horizon in [21, 63]:
                is_n, is_excess = lookup_period_cell(source, variant, regime, horizon, "in_sample")
                ho_n, ho_excess = lookup_period_cell(source, variant, regime, horizon, "holdout")
                is_n, ho_n = is_n or 0, ho_n or 0
                low_n = is_n < MIN_BUCKET_N or ho_n < MIN_BUCKET_N

                if low_n or is_excess is None or ho_excess is None:
                    flag = "low_n"
                elif abs(is_excess) < NEAR_ZERO_EXCESS:
                    flag = "inconclusive"
                else:
                    same_sign = np.sign(is_excess) == np.sign(ho_excess)
                    decay = abs(ho_excess) / abs(is_excess)
                    if same_sign and decay >= 0.5:
                        flag = "consistent"
                    elif same_sign:
                        flag = "directionally_consistent_but_weaker"
                    else:
                        flag = "sign_flipped"

                consistency_rows.append({"regime_source": source, "variant": variant, "regime": regime,
                                          "horizon": horizon, "in_sample_count": is_n, "holdout_count": ho_n,
                                          "in_sample_excess": is_excess, "holdout_excess": ho_excess,
                                          "consistency_flag": flag})

in_sample_vs_holdout_consistency = pd.DataFrame(consistency_rows)
print(in_sample_vs_holdout_consistency["consistency_flag"].value_counts())

consistency_flag
sign_flipped                           31
consistent                             14
directionally_consistent_but_weaker     7
low_n                                   4
Name: count, dtype: int64


Across all tested variant/regime/horizon cells, the consistency-flag counts are the central
empirical result of this notebook: **31 of 56 tested cells sign-flipped between in-sample and
holdout, 14 were fully `"consistent"`, and 21 total cells qualified as descriptive holdout
survivors** (`"consistent"` or `"directionally_consistent_but_weaker"` combined, at adequate
sample size in both periods; 4 further cells were excluded for low sample size in one period).
The majority of what looked interesting in-sample still did not hold its sign, let alone its
magnitude, once tested on data the framework never saw during construction — but a meaningful
minority (roughly 37%) did, which is enough to keep this from reading as pure noise.

## 4. Does Persistence Confirmation Improve Temporal Stability?

For each variant/regime/horizon cell, compare the `raw` and `confirmed_5d` consistency flags
side by side and identify which label set is preferred on temporal-stability grounds — never on
which one happened to show the larger in-sample excess, since that would just relocate the
overfitting problem from regime definition to label-set selection.

In [8]:
FLAG_RANK = {"consistent": 0, "directionally_consistent_but_weaker": 1,
             "inconclusive": 2, "sign_flipped": 3, "low_n": 4}

def consistency_row(source, variant, regime, horizon):
    row = in_sample_vs_holdout_consistency[
        (in_sample_vs_holdout_consistency["regime_source"] == source)
        & (in_sample_vs_holdout_consistency["variant"] == variant)
        & (in_sample_vs_holdout_consistency["regime"] == regime)
        & (in_sample_vs_holdout_consistency["horizon"] == horizon)
    ]
    return row.iloc[0] if len(row) else None


preference_counts = {"raw": 0, "confirmed_5d": 0, "neither": 0, "inconclusive": 0}
for variant in signal_variants:
    for regime in REGIME_STATES:
        for horizon in [21, 63]:
            raw_row = consistency_row("raw", variant, regime, horizon)
            c5_row = consistency_row("confirmed_5d", variant, regime, horizon)
            if raw_row is None or c5_row is None:
                continue
            raw_rank, c5_rank = FLAG_RANK[raw_row["consistency_flag"]], FLAG_RANK[c5_row["consistency_flag"]]
            if raw_rank < c5_rank:
                preference_counts["raw"] += 1
            elif c5_rank < raw_rank:
                preference_counts["confirmed_5d"] += 1
            elif raw_row["consistency_flag"] in ("sign_flipped", "inconclusive"):
                preference_counts["neither"] += 1
            else:
                preference_counts["inconclusive"] += 1

print("Preferred label set across variant/regime/horizon comparisons:")
for label, count in preference_counts.items():
    print(f"  {label:14s}: {count}")

Preferred label set across variant/regime/horizon comparisons:
  raw           : 9
  confirmed_5d  : 3
  neither       : 10
  inconclusive  : 2


Persistence confirmation was preferred on temporal-stability grounds in only **3 of 24**
variant/regime/horizon comparisons (raw preferred in 9, neither preferred in 10, inconclusive in
2). **Persistence confirmation did not show a clear, broad improvement in temporal stability** —
the raw same-date labels and the 5-day-confirmed labels performed about equally poorly out of
sample, with the confirmed labels additionally paying a real sample-size cost. Requiring a regime
to persist longer before trusting it is an intuitively reasonable filter, but it did not rescue
the framework's forward-return claims here.

## Temporal Holdout Verdict

**Overall G11 verdict: partially supported.**

A meaningful minority of descriptive relationships (21 of 56 tested cells) held their sign and a
meaningful share of their in-sample magnitude out of sample — enough that this is not a complete,
uniform failure. But the majority (31 of 56 cells) still sign-flipped, and persistence
confirmation — the one concrete refinement tested to address the flickery-regime concern — did
not meaningfully change that picture.

**G11 does not establish predictive power, causality, or strategy readiness.** What it does
establish, honestly:

- The framework is diagnostically coherent (Phase 1, Phase 2 structural checks) and produces some
  descriptive forward-return separation (Phase 2 G6).
- That separation does not clearly exceed what simple baselines already capture (Phase 2 G7).
- It does not meaningfully survive a stacked permutation-plus-HAC robustness check on its
  best-looking candidates (Phase 2 G8) — 1 of 13, a single near-boundary case, roughly what
  multiple-comparison risk alone would predict by chance.
- Persistence-confirming regimes does not fix this (G10–G11).
- A minority of relationships (21 of 56 cells) survive a genuine chronological holdout, which
  keeps this from being dismissible as pure noise — but a minority surviving is not the same as
  the framework working.

**Overall research-arc verdict: partially supported.** This is the honest final chapter of the
V&D Cloud research arc: a coherent, replicable diagnostic instrument whose forward-return claims
were tested at every stage a careful reader would ask about, and which did not clear the bar for
predictive validity or trade readiness at any of them. Treat the framework as diagnostic research
worth extending cautiously — not as a signal, and not as something to trade.